In [ ]:
import os
import random
import shutil
import yaml
from ultralytics import YOLO

# Xác định đường dẫn gốc của dự án (Project Root)
# Giả định file này nằm trong thư mục 'notebooks/', ta đi ngược ra 1 cấp
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(BASE_DIR, "data", "processed")

train_img = os.path.join(DATA_PATH, 'images', 'train')
train_lbl = os.path.join(DATA_PATH, 'labels', 'train')
val_img = os.path.join(DATA_PATH, 'images', 'val')
val_lbl = os.path.join(DATA_PATH, 'labels', 'val')

print(f"📂 Thư mục gốc dự án: {BASE_DIR}")
print(f"📊 Dữ liệu train tại: {train_img}")

In [ ]:
# 1. Tạo thư mục Val nếu chưa có
os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

# 2. Chia tập VAL chuẩn (20%) - Chỉ chạy nếu tập VAL trống
if len(os.listdir(val_img)) == 0:
    print("🗂️ Đang chia tập VAL...")
    imgs = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.png'))]
    num_val = int(len(imgs) * 0.2)
    for img in random.sample(imgs, num_val):
        shutil.move(os.path.join(train_img, img), os.path.join(val_img, img))
        lbl = os.path.splitext(img)[0] + '.txt'
        if os.path.exists(os.path.join(train_lbl, lbl)):
            shutil.move(os.path.join(train_lbl, lbl), os.path.join(val_lbl, lbl))
    print(f"✅ Đã chia xong {num_val} ảnh sang VAL.")

# 3. Nhân bản x10 cho Car (2) và Truck (5)
# Kiểm tra tiền tố để tránh nhân bản trùng lặp nhiều lần
if not any(f.startswith('local_aug_') for f in os.listdir(train_img)):
    print("⚖️ Đang nhân bản x10 cho Car và Truck...")
    for file in [f for f in os.listdir(train_lbl) if f.endswith('.txt')]:
        with open(os.path.join(train_lbl, file), 'r') as f:
            content = f.read()
            if any(cls in content for cls in ['2', '5']):
                img_ext = '.jpg' if os.path.exists(os.path.join(train_img, file.replace('.txt', '.jpg'))) else '.png'
                img_f = file.replace('.txt', img_ext)
                if os.path.exists(os.path.join(train_img, img_f)):
                    for i in range(10):
                        shutil.copy(os.path.join(train_img, img_f), os.path.join(train_img, f"local_aug_{i}_{img_f}"))
                        shutil.copy(os.path.join(train_lbl, file), os.path.join(train_lbl, f"local_aug_{i}_{file}"))
    print("✅ Hoàn tất cân bằng dữ liệu.")

In [ ]:
# 1. Ghi file data.yaml với đường dẫn tương đối chuẩn
data_yaml = {
    'path': DATA_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 6,
    'names': {0: 'NGUOI DI BO', 1: 'XE DAP', 2: 'O TO', 3: 'XE MAY', 4: 'XE BUYT', 5: 'XE TAI'}
}

yaml_file_path = os.path.join(DATA_PATH, 'data.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(data_yaml, f)

# 2. Huấn luyện Local
print("🚀 Khởi động YOLOv26 Nano on Local GPU...")
model = YOLO(os.path.join(BASE_DIR, 'yolo26n.pt')) 

model.train(
    data=yaml_file_path,
    epochs=150,
    imgsz=640,
    batch=-1,        # Tự động tối ưu theo GPU local
    project=os.path.join(BASE_DIR, 'runs'),
    name='SafeTruck_Local_V26',
    optimizer='AdamW',
    close_mosaic=20,
    cos_lr=True,
    device=0         # Mặc định dùng GPU đầu tiên
)